# Build GPT from scratch
这个笔记是跟随 [Karpath - Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=9) 用 PyTorch 编写一个基础版本的 GPT模型；本人早些年接触过机器学习相关知识，这个笔记算是从头捡起 Tranformer 和 PyTorch

## 1 Start with Bigram model
第一部分，我们下载测试数据并使用一个 Bigram model 来熟悉模型的构建和训练过程。

一个Bigram 模型仅根据当前Token对下一个Token进行预测：

$$P(x_{t+1} \mid x_t)$$

例如，在当前Token `deep`, t这个模型可能会生成下个Token `learning`.

### 1.1 下载测试数据并解析

In [ ]:
import urllib.request as ur

# 下载数据集
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

output = "input.txt"
ur.urlretrieve(url, output)

# Python 上下文协议，用于在代码块结束时自动执行清理操作，例如关闭文件
with open("input.txt", "r") as f:
    text = f.read()

print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


下载好了测试数据以后，需要将文本转化为数字从而进行计算和训练

In [6]:
# 我们先构建一个简单的字符字典
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("字符表大小:", vocab_size)

# 创建字符到索引的映射
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

# 测试映射是否正确
print("字符 'a' 的索引:", stoi['a'])
print(F"索引 {stoi['a']} 对应的字符:", itos[stoi['a']])

# 这样我们就可以借助stoi 和 itos完成编码和解码
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# 测试编码和解码
print("编码 'hello':", encode("hello"))
print(f"解码 {encode('hello')}:", decode(encode('hello')))

字符表大小: 65
字符 'a' 的索引: 39
索引 39 对应的字符: a
编码 'hello': [46, 43, 50, 50, 53]
解码 [46, 43, 50, 50, 53]: hello


接下来我们开始利用 PyTorch 构建一个 Bigram 模型并训练

In [ ]:
# 将训练数据转为 Tensor
import torch

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

block_size = 8
batch_size = 4

def get_batch(data="train"):
    if data == "train":
        data = train_data
    else:
        data = val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)


In [ ]:
# 创建 Bigram 模型
torch.manual_seed(42)
class Bigram(torch.nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # 将embedding维度设置为 size * size
        # 第 idx 个 token 对应的 embedding 向量就是 logits 中的第 idx 行
        self.token_embedding_table = torch.nn.Embedding(vocab_size, vocab_size)

    # 前向传播函数，输入 idx 和可选的 targets，返回 logits 和 loss
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # 将 logits 和 targets 展平成二维和一维，以便计算交叉熵损失
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = torch.nn.functional.cross_entropy(logits, targets)
        return logits, loss

    # 生成下一个 token 的函数，输入当前的 idx，返回下一个 token 的概率分布
    def inference(self, idx, max_new_tokens=10):
        # self 可以直接调用因为实现了 __call__ 函数
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]  # 只取最后一个时间步的 logits
            probs = torch.nn.functional.softmax(logits, dim=-1)

            # 根据概率分布采样下一个 token
            idx_next = torch.multinomial(probs, num_samples=1)
            # 将采样得到的 token 拼接到输入 idx 的末尾
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

bigram = Bigram(vocab_size)
logits, loss = bigram(xb, yb)
print(loss)
print(logits.shape)
        

In [ ]:
# 开始训练
optimizer = torch.optim.AdamW(bigram.parameters(), lr=1e-3)

eval_iters = 200
max_iters = 10000
eval_interval = 500

for steps in range(max_iters):
    xb, yb = get_batch('train')
    logits, loss = bigram(xb, yb)
    optimizer.zero_grad(set_to_none=True) # 将梯度置为 None 而不是 0，可以节省内存并加快训练
    loss.backward() # 反向传播计算梯度
    optimizer.step() # 更新模型参数

print(loss.item())
print('\n')

print(decode(bigram.inference(
    idx = torch.zeros((1, 1), dtype=torch.long),
    max_new_tokens = 100
)[0].tolist()))


In [ ]:
# 修改训练过程，记录每一步的loss并绘制图标
